# Recertification KYC — Pipeline GPU (vLLM) pour Qwen2.5-VL, InternVL, PaddleOCR-VL

## Pourquoi cette réécriture

La version précédente appelait `model.generate()` (transformers pur) **image par image**,
ce qui sous-utilise fortement le GPU (pas de batching, pas de paging mémoire, un seul flux
séquentiel). Pour un traitement en masse de dossiers KYC, on bascule sur des librairies
**conçues pour l'inférence GPU à haut débit** :

- **vLLM** pour Qwen2.5-VL et InternVL : *continuous batching* + *PagedAttention*, permet
  de traiter des dizaines/centaines d'images en parallèle sur le GPU au lieu d'une boucle séquentielle.
- **transformers + `torch.cuda` + `bfloat16` + `flash_attention_2`** pour PaddleOCR-VL
  (vLLM ne supporte pas toujours nativement les architectures VLM très récentes/custom —
  on force ici explicitement le GPU, la précision réduite, et l'attention optimisée).
- **Gestion explicite de la mémoire GPU** entre chaque modèle (`del`, `gc.collect()`,
  `torch.cuda.empty_cache()`, nettoyage vLLM) car les 3 modèles ne tiennent pas
  nécessairement ensemble en mémoire sur un seul GPU.

## Changement d'architecture du pipeline (traitement par phases, pas par client)

Plutôt que de traiter un client de A à Z avant de passer au suivant (ce qui forcerait à
garder les 3 modèles chargés simultanément), le pipeline traite maintenant **tous les
documents en une seule fois, phase par phase** :

1. **Phase 0 (CPU)** : dézippage, repérage des PDF, conversion en images, prétraitement
   (rotation/skew/débruitage) → tous les documents de tous les clients sont préparés en mémoire.
2. **Phase 1 (GPU, vLLM)** : chargement de Qwen2.5-VL, extraction schema-free **en un seul
   batch** sur toutes les images → déchargement du modèle.
3. **Phase 2 (GPU, vLLM)** : chargement d'InternVL, même traitement en batch → déchargement.
4. **Phase 3 (GPU, transformers)** : chargement de PaddleOCR-VL, transcription brute en
   batch (par lots) → déchargement.
5. **Phase 4 (CPU)** : réconciliation Qwen ↔ InternVL ↔ PaddleOCR-VL et cross-vérification
   d'identité entre `JUSTIFICATIF IDENTITE.PDF` et `JUSTIFICATIF DOMICILE.PDF`, par client.

Ce découpage maximise l'utilisation du GPU (un seul modèle chargé à la fois, à pleine
capacité de batching) au lieu de faire du "va-et-vient" entre 3 modèles par document.


In [ ]:
# Dépendances GPU nécessaires (à exécuter une fois dans l'environnement) :
# pip install vllm --extra-index-url https://download.pytorch.org/whl/cu121
# pip install "transformers>=4.46" accelerate flash-attn --no-build-isolation
# pip install pymupdf opencv-python-headless pillow pandas
import os
import re
import io
import gc
import json
import zipfile
import shutil
import unicodedata
import difflib
import math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from PIL import Image
import torch

assert torch.cuda.is_available(), "Aucun GPU CUDA detecte : ce notebook suppose un environnement GPU."
print(f"GPU detecte : {torch.cuda.get_device_name(0)}")
print(f"Memoire GPU totale : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")


## 1. Configuration : chemins ModelHub, archive ZIP, réglages GPU

In [ ]:
MODEL_PATHS = {
    "qwen_vlm": "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-instruct/main",
    "internvl": "/domino/edv/modelhub/ModelHub-model-huggingface-OpenGVLab/InternVL2_5-8B/main",
    "paddleocr_vl": "/domino/edv/modelhub/ModelHub-model-huggingface-PaddlePaddle/PaddleOCR-VL/main",
}

ZIP_PATH = "/mnt/user-data/uploads/dossiers_clients.zip"   # a adapter
EXTRACT_DIR = "/home/work/kyc_extraction"
OUTPUT_DIR = "/mnt/user-data/outputs"

TARGET_FILES = {
    "id_card": ["justificatifidentite", "carteidentite", "cni", "idcard", "national id"],
    "residence": ["justificatifdomicile", "residence", "proofofaddress", "residencepermit"],
}

# Reglages GPU / vLLM
GPU_MEMORY_UTILIZATION = 0.85   # part de la memoire GPU allouee a chaque instance vLLM
MAX_MODEL_LEN = 4096
PADDLEOCR_BATCH_SIZE = 8         # taille de lot pour l'inference transformers batchee

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Phase 0 (CPU) — décompression, repérage des PDF, conversion en images, prétraitement

In [ ]:
def extract_zip(zip_path: str, dest_dir: str) -> str:
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    return dest_dir


def list_client_folders(root_dir: str) -> dict:
    entries = [e for e in Path(root_dir).iterdir() if e.is_dir()]
    if len(entries) == 1 and any(sub.is_dir() for sub in entries[0].iterdir()):
        entries = [e for e in entries[0].iterdir() if e.is_dir()]
    return {e.name: str(e) for e in entries}


def normalize_filename(name: str) -> str:
    name = os.path.splitext(name)[0]
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[\s_\-]+", "", name.lower())


def find_target_pdfs(client_folder: str) -> dict:
    found = {"id_card": None, "residence": None}
    for f in Path(client_folder).glob("*.[pP][dD][fF]"):
        norm = normalize_filename(f.name)
        for doc_key, aliases in TARGET_FILES.items():
            if found[doc_key] is not None:
                continue
            if any(normalize_filename(alias) in norm for alias in aliases):
                found[doc_key] = str(f)
    return found


def pdf_to_images(pdf_path: str, dpi: int = 300) -> list:
    images = []
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(matrix=matrix)
        img_bytes = pix.tobytes("png")
        img_arr = np.frombuffer(img_bytes, dtype=np.uint8)
        img_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        images.append(img_bgr)
    doc.close()
    return images


def denoise_and_enhance(img_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def detect_coarse_rotation_osd(img_bgr: np.ndarray) -> int:
    try:
        import pytesseract
        osd = pytesseract.image_to_osd(img_bgr)
        angle = int([l for l in osd.split("\n") if "Rotate" in l][0].split(":")[1].strip())
        return angle
    except Exception as e:
        print(f"[WARN] OSD indisponible ({e}), rotation grossiere non detectee.")
        return 0


def rotate_image(img_bgr: np.ndarray, angle: float) -> np.ndarray:
    if angle == 0:
        return img_bgr
    (h, w) = img_bgr.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = np.abs(M[0, 0]); sin = np.abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    return cv2.warpAffine(img_bgr, M, (new_w, new_h), borderValue=(255, 255, 255))


def detect_and_correct_skew(img_bgr: np.ndarray, max_angle: float = 15.0) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100,
                             minLineLength=gray.shape[1] // 4, maxLineGap=20)
    if lines is None:
        return img_bgr
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
        if -max_angle <= angle <= max_angle:
            angles.append(angle)
    if not angles:
        return img_bgr
    median_angle = float(np.median(angles))
    return img_bgr if abs(median_angle) < 0.5 else rotate_image(img_bgr, median_angle)


def preprocess_image(img_bgr: np.ndarray) -> np.ndarray:
    img_bgr = rotate_image(img_bgr, detect_coarse_rotation_osd(img_bgr))
    img_bgr = detect_and_correct_skew(img_bgr)
    img_bgr = denoise_and_enhance(img_bgr)
    return img_bgr


@dataclass
class DocImage:
    client_id: str
    doc_type: str          # "id_card" ou "residence"
    page_index: int
    image: Image.Image     # PIL RGB, deja pretraitee


def build_document_index(zip_path: str) -> list:
    """Phase 0 complete : dezippage -> reperage PDF -> conversion -> pretraitement.
    Retourne une liste plate de DocImage prets pour l'inference GPU en batch."""
    root = extract_zip(zip_path, EXTRACT_DIR)
    client_folders = list_client_folders(root)
    print(f"{len(client_folders)} dossiers clients detectes.")

    doc_images = []
    missing = []
    for cid, folder in client_folders.items():
        pdfs = find_target_pdfs(folder)
        if pdfs["id_card"] is None or pdfs["residence"] is None:
            missing.append({"client_id": cid, **{k: (v is not None) for k, v in pdfs.items()}})
        for doc_type, pdf_path in pdfs.items():
            if pdf_path is None:
                continue
            pages = pdf_to_images(pdf_path)
            for i, page_bgr in enumerate(pages):
                corrected = preprocess_image(page_bgr)
                img_pil = Image.fromarray(cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB))
                doc_images.append(DocImage(client_id=cid, doc_type=doc_type, page_index=i, image=img_pil))

    print(f"{len(doc_images)} pages preparees pour l'inference GPU.")
    if missing:
        print(f"[INFO] {len(missing)} clients avec un document manquant (voir df_missing).")
    return doc_images, pd.DataFrame(missing), client_folders


doc_images, df_missing, client_folders = build_document_index(ZIP_PATH)
df_missing.head(10)


## 3. Prompt d'extraction schema-free (commun à Qwen2.5-VL et InternVL)

Aucun champ prédéfini, aucun schéma par type de document — le modèle liste lui-même
les champs observés.


In [ ]:
OPEN_EXTRACTION_PROMPT = """Tu es un expert en analyse de documents d'identite et justificatifs
administratifs, de tous pays et dans toutes les langues.

Observe attentivement ce document (il peut etre mal scanne, etre une photocopie, legerement
de travers, ou dans une langue que tu ne maitrises pas parfaitement).

Ne suppose AUCUN format ni AUCUNE liste de champs predefinie : identifie uniquement les
champs reellement visibles sur CE document, tels qu'ils apparaissent.

Reponds UNIQUEMENT avec un objet JSON valide, structure suivante (structure fixe,
mais le CONTENU des champs est entierement libre) :

{
  "type_document_detecte": "ex: carte d'identite nationale, passeport, titre de sejour, facture, attestation de domicile, ...",
  "pays_emetteur_estime": "",
  "langue_document": "",
  "champs": [
    {"libelle_original": "texte tel qu'il apparait sur le document", "libelle_normalise_fr": "traduction/normalisation en francais, snake_case", "valeur": "valeur lue"},
    ...
  ],
  "remarques": "toute anomalie constatee : illisible, incoherent, incomplet, etc."
}

Liste TOUS les champs que tu peux lire, sans te limiter a un sous-ensemble. Ne donne
aucun texte en dehors du JSON."""


def safe_json_parse(text: str) -> dict:
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return {"_parse_error": True, "_raw_output": text}
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return {"_parse_error": True, "_raw_output": text}


def free_gpu_memory(*objs):
    """Libere explicitement la memoire GPU entre deux phases (indispensable pour
    faire tenir 3 gros modeles successivement sur un seul GPU)."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel()
        destroy_distributed_environment()
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"Memoire GPU libre apres nettoyage : {torch.cuda.mem_get_info()[0] / 1e9:.1f} Go")


## 4. Phase 1 (GPU/vLLM) — Extraction schema-free avec Qwen2.5-VL, en un seul batch

vLLM prend en charge le *continuous batching* : toutes les images sont soumises en une
seule fois, le moteur gère lui-même l'ordonnancement optimal sur le GPU (bien plus rapide
qu'une boucle `for image in images: model.generate(...)`).


In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

sampling_params = SamplingParams(temperature=0.0, max_tokens=768)

def run_vllm_extraction(model_path: str, doc_images: list, prompt_text: str,
                         limit_mm_per_prompt: int = 1) -> list:
    """Charge un VLM via vLLM et lance l'extraction en un seul appel batch sur
    toutes les images fournies. Retourne la liste des dicts JSON parses, dans
    le meme ordre que doc_images."""
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

    requests = []
    for doc in doc_images:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": doc.image},
                {"type": "text", "text": prompt_text},
            ],
        }]
        prompt_str = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        requests.append({"prompt": prompt_str, "multi_modal_data": {"image": doc.image}})

    llm = LLM(
        model=model_path,
        trust_remote_code=True,
        dtype="bfloat16",
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
        limit_mm_per_prompt={"image": limit_mm_per_prompt},
    )

    outputs = llm.generate(requests, sampling_params)
    parsed = [safe_json_parse(o.outputs[0].text) for o in outputs]

    free_gpu_memory(llm, processor)
    return parsed


print("Phase 1/3 : extraction avec Qwen2.5-VL (vLLM, batch complet)...")
qwen_outputs = run_vllm_extraction(MODEL_PATHS["qwen_vlm"], doc_images, OPEN_EXTRACTION_PROMPT)
print(f"{len(qwen_outputs)} resultats Qwen2.5-VL obtenus.")


## 5. Phase 2 (GPU/vLLM) — Second avis avec InternVL, en un seul batch

Même principe, avec le modèle InternVL déjà présent sur le ModelHub. Qwen2.5-VL a été
déchargé de la mémoire GPU à la fin de la phase précédente (`free_gpu_memory`).


In [ ]:
print("Phase 2/3 : extraction avec InternVL (vLLM, batch complet)...")
internvl_outputs = run_vllm_extraction(MODEL_PATHS["internvl"], doc_images, OPEN_EXTRACTION_PROMPT)
print(f"{len(internvl_outputs)} resultats InternVL obtenus.")


## 6. Phase 3 (GPU/transformers) — Transcription brute avec PaddleOCR-VL, par lots

vLLM ne supporte pas nécessairement encore l'architecture exacte de PaddleOCR-VL selon la
version livrée sur le ModelHub ; on utilise donc `transformers` mais avec les optimisations
GPU explicites : `torch_dtype=bfloat16`, `device_map="cuda"`, `attn_implementation="flash_attention_2"`
(si installé), et un **batching manuel par lots** (`PADDLEOCR_BATCH_SIZE`) plutôt qu'une
image à la fois.


In [ ]:
from transformers import AutoModel

def load_paddleocr_vl(model_path: str):
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    try:
        model = AutoModel.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            device_map="cuda",
            trust_remote_code=True,
            attn_implementation="flash_attention_2",
        )
    except Exception as e:
        print(f"[WARN] flash_attention_2 indisponible ({e}), repli sur l'attention par defaut.")
        model = AutoModel.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            device_map="cuda",
            trust_remote_code=True,
        )
    model.eval()
    return processor, model


def run_paddleocr_vl_batch(processor, model, images: list, prompt_text: str = "OCR:",
                            batch_size: int = PADDLEOCR_BATCH_SIZE) -> list:
    """Transcription brute par lots sur GPU. Le batching reel depend du support du
    processor pour plusieurs images par appel ; a defaut, ajuster batch_size=1."""
    results = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            chunk = images[start:start + batch_size]
            messages_batch = [
                [{"role": "user", "content": [{"type": "image", "image": img},
                                               {"type": "text", "text": prompt_text}]}]
                for img in chunk
            ]
            prompts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                       for m in messages_batch]
            inputs = processor(text=prompts, images=chunk, return_tensors="pt", padding=True).to(model.device)
            output_ids = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
            decoded = processor.batch_decode(
                output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )
            results.extend([t.strip() for t in decoded])
            print(f"  PaddleOCR-VL : {min(start + batch_size, len(images))}/{len(images)} pages traitees")
    return results


print("Phase 3/3 : transcription brute avec PaddleOCR-VL (transformers, GPU, par lots)...")
paddle_processor, paddle_model = load_paddleocr_vl(MODEL_PATHS["paddleocr_vl"])
paddle_raw_texts = run_paddleocr_vl_batch(paddle_processor, paddle_model, [d.image for d in doc_images])
free_gpu_memory(paddle_model, paddle_processor)
print(f"{len(paddle_raw_texts)} transcriptions PaddleOCR-VL obtenues.")


## 7. Phase 4 (CPU) — Réconciliation Qwen ↔ InternVL ↔ PaddleOCR-VL, par client

Toutes les inférences GPU sont terminées ; cette phase est purement CPU (comparaisons
de texte, agrégation par client). On associe chaque résultat à son `DocImage` d'origine
via l'index de liste (même ordre que `doc_images`).


In [ ]:
def normalize_text(v) -> str:
    if v is None:
        return ""
    v = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", v.strip().lower())


def normalize_no_space(v) -> str:
    return re.sub(r"\s+", "", normalize_text(v))


def estimate_text_quality(raw_text: str) -> float:
    if not raw_text:
        return 0.0
    alnum_ratio = sum(c.isalnum() for c in raw_text) / max(len(raw_text), 1)
    length_score = min(len(raw_text) / 500.0, 1.0)
    return round(0.6 * alnum_ratio + 0.4 * length_score, 3)


def match_fields(champs_a: list, champs_b: list, threshold: float = 0.72) -> list:
    used_b = set()
    matches = []
    for fa in champs_a:
        label_a = normalize_text(fa.get("libelle_normalise_fr"))
        best_j, best_ratio = None, 0.0
        for j, fb in enumerate(champs_b):
            if j in used_b:
                continue
            label_b = normalize_text(fb.get("libelle_normalise_fr"))
            ratio = difflib.SequenceMatcher(None, label_a, label_b).ratio()
            if ratio > best_ratio:
                best_ratio, best_j = ratio, j
        if best_j is not None and best_ratio >= threshold:
            used_b.add(best_j)
            matches.append((fa, champs_b[best_j], best_ratio))
    return matches


def verify_values_against_ocr(champs: list, ocr_raw_text: str) -> dict:
    ocr_norm = normalize_no_space(ocr_raw_text)
    verified, unverified = [], []
    for c in champs:
        val = normalize_no_space(c.get("valeur"))
        if not val:
            continue
        (verified if (len(val) >= 3 and val in ocr_norm) else unverified).append(c)
    total = len(verified) + len(unverified)
    rate = (len(verified) / total) if total > 0 else None
    return {"taux_verification_ocr": round(rate, 2) if rate is not None else None,
            "champs_non_retrouves_dans_ocr": unverified}


def reconcile_extractions(champs_a: list, champs_b: list, ocr_raw_text: str) -> dict:
    matches = match_fields(champs_a, champs_b)
    disagreements, agreements = [], []
    for fa, fb, sim in matches:
        va, vb = normalize_text(fa.get("valeur")), normalize_text(fb.get("valeur"))
        entry = {"libelle": fa.get("libelle_normalise_fr"), "valeur_qwen": fa.get("valeur"),
                  "valeur_internvl": fb.get("valeur"), "similarite_libelle": round(sim, 2)}
        (agreements if va == vb else disagreements).append(entry)

    n_matched = len(matches)
    n_total = max(len(champs_a), len(champs_b), 1)
    coverage = n_matched / n_total
    agreement_rate = (len(agreements) / n_matched) if n_matched > 0 else 0.0

    ocr_verification = verify_values_against_ocr(champs_a, ocr_raw_text)
    ocr_rate = ocr_verification["taux_verification_ocr"] if ocr_verification["taux_verification_ocr"] is not None else 0.5

    confidence = round(coverage * agreement_rate * (0.5 + 0.5 * ocr_rate), 3)
    statut = "certifie" if (confidence >= 0.5 and len(disagreements) == 0 and ocr_rate >= 0.7) else "a_revoir_manuellement"

    return {
        "champs_qwen": champs_a, "champs_internvl": champs_b,
        "champs_en_accord": agreements, "champs_en_desaccord": disagreements,
        "couverture_appariement": round(coverage, 2), "taux_accord": round(agreement_rate, 2),
        "verification_paddleocr_vl": ocr_verification, "confiance": confidence, "statut": statut,
    }


NAME_LABEL_HINTS = ["nom", "prenom", "nom complet", "name", "surname", "full name",
                    "nombre", "apellido", "nachname", "vorname", "cognome", "nome"]

def extract_name_like_fields(champs: list) -> list:
    found = []
    for c in champs:
        label = normalize_text(c.get("libelle_normalise_fr")) + " " + normalize_text(c.get("libelle_original"))
        if any(hint in label for hint in NAME_LABEL_HINTS):
            found.append(c.get("valeur"))
    return found


def cross_check_identity(id_card_champs: list, residence_champs: list) -> dict:
    names_id = [normalize_text(n) for n in extract_name_like_fields(id_card_champs) if n]
    names_res = [normalize_text(n) for n in extract_name_like_fields(residence_champs) if n]
    if not names_id or not names_res:
        return {"verifiable": False, "coherent": None,
                "detail": "champs nominatifs introuvables sur l'un des deux documents"}
    best_ratio = 0.0
    for n1 in names_id:
        for n2 in names_res:
            best_ratio = max(best_ratio, difflib.SequenceMatcher(None, n1, n2).ratio())
    return {"verifiable": True, "coherent": best_ratio >= 0.75, "similarite_max": round(best_ratio, 2),
            "noms_carte_identite": names_id, "noms_residence": names_res}


## 8. Agrégation par client et export final

In [ ]:
def aggregate_by_client(doc_images: list, qwen_outputs: list, internvl_outputs: list,
                         paddle_raw_texts: list, client_folders: dict) -> tuple:
    grouped = {}
    for idx, doc in enumerate(doc_images):
        key = (doc.client_id, doc.doc_type)
        grouped.setdefault(key, {"qwen_champs": [], "internvl_champs": [], "ocr_text": [],
                                  "type_document_detecte": None, "pays": None, "langue": None,
                                  "n_pages": 0})
        g = grouped[key]
        g["n_pages"] += 1
        qwen_out = qwen_outputs[idx]
        internvl_out = internvl_outputs[idx]
        g["qwen_champs"].extend(qwen_out.get("champs", []) if isinstance(qwen_out.get("champs"), list) else [])
        g["internvl_champs"].extend(internvl_out.get("champs", []) if isinstance(internvl_out.get("champs"), list) else [])
        g["ocr_text"].append(paddle_raw_texts[idx])
        g["type_document_detecte"] = g["type_document_detecte"] or qwen_out.get("type_document_detecte")
        g["pays"] = g["pays"] or qwen_out.get("pays_emetteur_estime")
        g["langue"] = g["langue"] or qwen_out.get("langue_document")

    doc_results = {}
    for (cid, doc_type), g in grouped.items():
        full_ocr_text = "\n".join(g["ocr_text"])
        reconciliation = reconcile_extractions(g["qwen_champs"], g["internvl_champs"], full_ocr_text)
        doc_results[(cid, doc_type)] = {
            "n_pages": g["n_pages"], "type_document_detecte": g["type_document_detecte"],
            "pays_emetteur_estime": g["pays"], "langue_document": g["langue"],
            "qualite_transcription_paddleocr_vl": round(float(np.mean([estimate_text_quality(t) for t in g["ocr_text"]])), 3),
            "transcription_paddleocr_vl": full_ocr_text,
            **reconciliation,
        }

    records = []
    for cid in client_folders:
        id_res = doc_results.get((cid, "id_card"), {"statut": "fichier_absent"})
        res_res = doc_results.get((cid, "residence"), {"statut": "fichier_absent"})

        if (cid, "id_card") in doc_results and (cid, "residence") in doc_results:
            cross_id = cross_check_identity(id_res.get("champs_qwen", []), res_res.get("champs_qwen", []))
        else:
            cross_id = {"verifiable": False, "detail": "un des deux documents est absent"}

        statuts = [id_res.get("statut"), res_res.get("statut")]
        identite_ok = cross_id.get("coherent", True)
        statut_global = ("a_revoir_manuellement"
                          if ("fichier_absent" in statuts or "a_revoir_manuellement" in statuts or identite_ok is False)
                          else "certifie")

        records.append({
            "client_id": cid,
            "justificatif_identite": id_res,
            "justificatif_domicile": res_res,
            "cross_verification_identite": cross_id,
            "statut_global": statut_global,
        })

    return pd.DataFrame(records), records


df_resultats, records_bruts = aggregate_by_client(
    doc_images, qwen_outputs, internvl_outputs, paddle_raw_texts, client_folders
)

df_resultats[["client_id", "statut_global"]]


In [ ]:
df_resultats.to_csv(f"{OUTPUT_DIR}/resultats_recertification_clients.csv", index=False)
with open(f"{OUTPUT_DIR}/resultats_recertification_clients.json", "w", encoding="utf-8") as f:
    json.dump(records_bruts, f, ensure_ascii=False, indent=2)
print(f"Resultats exportes dans {OUTPUT_DIR}")
df_resultats


## Notes de production (spécifiques GPU)

- **Mémoire GPU insuffisante pour un modèle 7-8B** : réduire `GPU_MEMORY_UTILIZATION`,
  activer la quantization vLLM (`quantization="awq"` ou `"bitsandbytes"` selon le
  checkpoint disponible), ou réduire `MAX_MODEL_LEN`.
- **Multi-GPU** : ajouter `tensor_parallel_size=N` dans `LLM(...)` pour répartir un modèle
  sur plusieurs GPU si un seul GPU ne suffit pas.
- **PaddleOCR-VL non compatible vLLM** : si une version future du checkpoint est prise en
  charge par vLLM, migrer `run_paddleocr_vl_batch` vers `run_vllm_extraction` pour un gain
  de débit supplémentaire (le code est volontairement isolé pour faciliter cette migration).
- **`flash_attention_2`** nécessite `pip install flash-attn --no-build-isolation` et une
  version CUDA compatible ; en son absence, le code se rabat automatiquement sur
  l'attention standard (voir le `try/except` dans `load_paddleocr_vl`).
- **Ordre des phases** : ne pas paralléliser le chargement des 3 modèles — les charger et
  décharger séquentiellement comme fait ici évite les erreurs `CUDA out of memory` sur un
  GPU unique.
- **Traçabilité KYC** : le JSON exporté conserve les champs bruts Qwen/InternVL et la
  transcription PaddleOCR-VL complète pour audit, même en cas de certification automatique.
